In [ ]:
''' 
# This should be all nessesary installs for the notebook to run

!pip install torch
!pip install numpy
!pip install plotly
!pip install scipy
!pip install tqdm
!pip install matplotlib
!pip install --upgrade nbconvert
'''

In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
import numpy as np
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import qmc
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm
import matplotlib.pyplot as plt

file_path10 = fr'C:\Users\marcs\Desktop\Github\Hooke_hole-coupled-4-2-r10.csv'
file_path25 = fr'C:\Users\marcs\Desktop\Github\Hooke_hole-coupled-4-2-r025.csv'
file_path40 = fr'C:\Users\marcs\Desktop\Github\Hooke_hole-coupled-4-2-r040.csv'

# Applied outer stresses
N1 = 4
N2 = 2
# Quadratic plate length
L = 1.0
# Lame constants
MU = 10
LBD = 20
# Elliptical axis in x direction
Rx = 0.1
Ry = 0.1
# Scaling factor
X = 2
# Edge samples
N = 50 * X
# Number of collocation points
M = 500 * X**2
# Epochs
EPOCHS = 200000
# Learning rate
LR = 0.001
# Gain parameter for initialization
GAIN = 0.5
# Scheduler step width
STEP = int(5000)
# Gamma factor of scheduler
GAMMA = 0.975
# Number of hidden neurons
HN = 50
# Number of hidden layers
LAYERS = 5
# Fourier features
FEATURES = 100
# Weight of PDE loss
W_PDE, W_DIR, W_NEU, W_HOLE = 1.0, 1.0, 1.0, 1.0
# Weight update factor
ALPHA = 0.9
# Variance
SIGMA = 1.0
# precision
DTYPE = torch.float32
# Optimized weights using gradient based adaptive method for Rx, Ry = 0.1, 0.1
W_DIR=1.61e+05
W_NEU=1.96e+02
W_PDE=1.09e+00
W_HOLE=1.85e+01


# DeepONet specific parameters
ITERATIONS = 10  # Number of iterations to average over
EPOCHS = int(EPOCHS/(ITERATIONS)) #10er minibatch
Longer_training_factor = 25
STEP = int((STEP * Longer_training_factor)/(ITERATIONS))
Global_Epoch = 0

Using device: cuda


# Sample and create the domain

In [2]:
def sample_rx_ry():
    Rx = 0.1 + 0.3 * torch.rand(1).item()
    Ry = 0.1 + 0.3 * torch.rand(1).item()
    return Rx, Ry

def sample_domain():
    Rx, Ry = sample_rx_ry()
    # Create collocation points
    points = qmc.LatinHypercube(d=2).random(M)
    mask = (((points[:, 0] ** 2) / (Rx**2)) + ((points[:, 1] ** 2) / (Ry**2))) > 1
    collocation = torch.tensor(points[mask], dtype=DTYPE, device=device)

    # Top boundary
    x_top = L * torch.tensor(qmc.LatinHypercube(d=1).random(N), dtype=DTYPE, device=device)
    y_top = L * torch.ones((N, 1), dtype=DTYPE, device=device)
    top = torch.column_stack([x_top, y_top])

    # Right boundary
    x_right = L * torch.ones((N, 1), dtype=DTYPE, device=device)
    y_right = L * torch.tensor(qmc.LatinHypercube(d=1).random(N), dtype=DTYPE, device=device)
    right = torch.column_stack([x_right, y_right])

    # Bottom boundary
    NN = int(N * (L - Rx) / L)
    rand_samp = qmc.LatinHypercube(d=1).random(NN)
    x_bottom = Rx + (L - Rx) * torch.tensor(rand_samp, dtype=DTYPE, device=device)
    y_bottom = torch.zeros((NN, 1), dtype=DTYPE, device=device)
    bottom = torch.column_stack([x_bottom, y_bottom])

    # Left boundary
    NN = int(N * (L - Ry) / L)
    rand_samp = qmc.LatinHypercube(d=1).random(NN)
    x_left = torch.zeros((NN, 1), dtype=DTYPE, device=device)
    y_left = Ry + (L - Ry) * torch.tensor(rand_samp, dtype=DTYPE, device=device)
    left = torch.column_stack([x_left, y_left])

    # Hole boundary
    num_hole_points = N #int(N * np.pi * Rx / L)
    rand_samp = qmc.LatinHypercube(d=1).random(num_hole_points).ravel()
    phi = 0.5 * np.pi * torch.tensor(rand_samp, dtype=DTYPE, device=device) 
    x_hole = Rx * torch.cos(phi)
    y_hole = Ry * torch.sin(phi)
    
    n_hole = torch.stack([-Ry * torch.cos(phi), -Rx * torch.sin(phi)]).T
    n_hole = n_hole / torch.linalg.norm(n_hole, axis=1)[:, None]
    hole = torch.column_stack([x_hole, y_hole])

    return collocation, top, bottom, left, right, hole, n_hole, Rx, Ry

collocation, top, bottom, left, right, hole, n_hole, Rx, Ry = sample_domain()

# Create a subplot with 1 row and 5 columns
fig = make_subplots(rows=1, cols=5, horizontal_spacing=0.02)

for i in range(5):
    with torch.inference_mode():
        collocation, top, bottom, left, right, hole, n_hole, Rx, Ry = sample_domain()
        mode = "markers"
        gray = dict(color="#C9C5BC")
        green = dict(color="#006561")
        black = dict(color="black")
        
        # Convert all tensors to CPU numpy arrays
        hole_cpu = hole.cpu().numpy() if hole.is_cuda else hole.numpy()
        n_hole_cpu = n_hole.cpu().numpy() if n_hole.is_cuda else n_hole.numpy()
        collocation_cpu = collocation.cpu().numpy() if collocation.is_cuda else collocation.numpy()
        top_cpu = top.cpu().numpy() if top.is_cuda else top.numpy()
        bottom_cpu = bottom.cpu().numpy() if bottom.is_cuda else bottom.numpy()
        left_cpu = left.cpu().numpy() if left.is_cuda else left.numpy()
        right_cpu = right.cpu().numpy() if right.is_cuda else right.numpy()
        
        # Create the quiver plot
        quiver_fig = ff.create_quiver(
            hole_cpu[:, 0], 
            hole_cpu[:, 1], 
            n_hole_cpu[:, 0], 
            n_hole_cpu[:, 1], 
            marker=black
        )
        
        # Add all traces to the appropriate subplot
        fig.add_trace(quiver_fig.data[0], row=1, col=i+1)
        fig.add_trace(go.Scatter(x=collocation_cpu[:, 0], y=collocation_cpu[:, 1], mode=mode, marker=gray), row=1, col=i+1)
        fig.add_trace(go.Scatter(x=top_cpu[:, 0], y=top_cpu[:, 1], mode=mode, marker=black), row=1, col=i+1)
        fig.add_trace(go.Scatter(x=bottom_cpu[:, 0], y=bottom_cpu[:, 1], mode=mode, marker=green), row=1, col=i+1)
        fig.add_trace(go.Scatter(x=left_cpu[:, 0], y=left_cpu[:, 1], mode=mode, marker=green), row=1, col=i+1)
        fig.add_trace(go.Scatter(x=right_cpu[:, 0], y=right_cpu[:, 1], mode=mode, marker=black), row=1, col=i+1)
        fig.add_trace(go.Scatter(x=hole_cpu[:, 0], y=hole_cpu[:, 1], mode=mode, marker=black), row=1, col=i+1)

fig.update_layout(
    template="none",
    width=2000,
    height=400,
    margin=dict(l=0, r=0, b=0, t=0),
    showlegend=False,
)

# Hide axes for all subplots
for i in range(5):
    fig.update_xaxes(visible=False, row=1, col=i+1)
    fig.update_yaxes(visible=False, row=1, col=i+1)
    fig.update_yaxes(scaleanchor="x", row=1, col=i+1)

fig.show()

# Create Branch and Trunk Network and combine them via the dot product

In [ ]:
# Branch Network (encodes geometry parameters Rx, Ry)
class BranchNet(torch.nn.Module):
    def __init__(self):
        super(BranchNet, self).__init__()
        # Hidden layers
        self.layers = torch.nn.ModuleList([
            torch.nn.Linear(2, HN, device=device).type(DTYPE)  # input: [Batch, 2] (Rx, Ry) Batch ist 1
        ])
        for _ in range(LAYERS - 1):
            self.layers.append(torch.nn.Linear(HN, HN).type(DTYPE))

        # Output: [Batch, HN * 2]
        # - HN = number of latent features (basis functions)
        # - 2 = displacement vector components (u_x, u_y)
        self.output_layer = torch.nn.Linear(HN, HN * 2, device=device).type(DTYPE) # es müssen 2x mehr outputs sein weil 2 komponenten der verschiebung

        # Xavier init
        for layer in self.layers:
            torch.nn.init.xavier_uniform_(layer.weight)

    def forward(self, r_params):
        """
        Input:
            r_params: [Batch, 2]  (Rx, Ry for each sample)
        Output:
            [Batch, HN, 2] 
            - dimension HN = latent basis size
            - 2 = displacement components (u_x, u_y)
        """
        x = r_params
        for layer in self.layers:
            x = torch.tanh(layer(x))
        return self.output_layer(x).reshape(-1, HN, 2) # Umwandlung von [Batch, HN*2] zu [Batch, HN, 2] damit später die einsum funktioniert


# Trunk Network (encodes spatial coordinates X, Y)
class TrunkNet(torch.nn.Module):
    def __init__(self):
        super(TrunkNet, self).__init__()
        # Random Fourier feature embedding for input coords [Batch, 2]
        self.B = torch.normal(0.0, SIGMA, size=(2, FEATURES), dtype=DTYPE, device=device)

        # Hidden layers
        self.layers = torch.nn.ModuleList([
            torch.nn.Linear(2 * FEATURES, HN, device=device).type(DTYPE) 
        ])
        for _ in range(LAYERS - 1):
            self.layers.append(torch.nn.Linear(HN, HN).type(DTYPE))

        # Output: [Batch, HN * 2]
        self.output_layer = torch.nn.Linear(HN, HN * 2, device=device).type(DTYPE)

        # Xavier init with scaling
        for layer in self.layers:
            torch.nn.init.xavier_uniform_(layer.weight, gain=GAIN)
        torch.nn.init.xavier_uniform_(self.output_layer.weight, gain=GAIN)

    def forward(self, x):
        """
        Input:
            x: [Batch, 2] (spatial coordinates X, Y)
        Output:
            [Batch, HN, 2]
            - dimension HN = latent basis size
            - 2 = displacement components (u_x, u_y)
        """
        # Fourier feature mapping: [Batch, 2] -> [Batch, 2*FEATURES]
        x = torch.cat([
            torch.sin(2 * np.pi * x @ self.B),
            torch.cos(2 * np.pi * x @ self.B)
        ], dim=-1)

        for layer in self.layers:
            x = torch.tanh(layer(x))
        return self.output_layer(x).reshape(-1, HN, 2) # Umwandlung von [Batch, HN*2] zu [Batch, HN, 2] damit später die einsum funktioniert

# DeepONet (combines Branch + Trunk)
class DeepONet(torch.nn.Module):
    def __init__(self, trunk_net, branch_net):
        super().__init__()
        self.trunk_net = trunk_net
        self.branch_net = branch_net

    def forward(self, x, r_params):
        """
        Input:
            x: [N, 2]          -> evaluation coordinates
            r_params: [B, 2]   -> geometry parameters (Rx, Ry)
        Output:
            [N, B, D] -> Displacement Ux and Uy [1, 1, 2]
            - N = number of evaluation points = 1
            - B = number of branch inputs = 1
            - D = number of output dimensions Ux and Uy = 2
        """
        trunk_output = self.trunk_net(x)          # [N, HN, 2] 
        branch_output = self.branch_net(r_params) # [B, HN, 2] 

        # Contract along latent dimension HN
        out = torch.einsum("nhd,bhd->nbd", trunk_output, branch_output).squeeze() # h = HN
        return out # output ist nun Ux und Uy


# Initialize networks
trunk_net = TrunkNet().to(device)
branch_net = BranchNet().to(device)
net = DeepONet(trunk_net, branch_net).to(device)

# Optimizer, Scheduler and Loss function
optimizer = torch.optim.Adam(list(trunk_net.parameters()) + list(branch_net.parameters()), lr=LR)
scheduler = StepLR(optimizer, step_size=STEP, gamma=GAMMA)
mse = torch.nn.MSELoss()

# Physics PDE + BCs

In [ ]:
def deformation_gradient(x, r_params):
    def forward_fn(x_): # Das ist damit es mit der jacrev Funktion funktioniert weil die nur 1 Argument erlaubt, dadurch ist r_params fixed für die Iteration
        return net(x_, r_params)
    dudx = torch.func.jacrev(forward_fn)(x).squeeze() # squeeze wegen der extra dimension welche beim PINN nicht da ist
    return torch.eye(2, dtype=DTYPE, device=device) + dudx

def first_piola_kirchhoff_stress(x, r_params):
    F = deformation_gradient(x, r_params)
    J = torch.det(F)
    return MU * F + (LBD * torch.log(J) - MU) * torch.inverse(F).T

def sigma(x, r_params):
    F = deformation_gradient(x, r_params)
    J = torch.det(F)
    P = first_piola_kirchhoff_stress(x, r_params)
    return (1 / J) * P @ F.T

def pde_residual(x, r_params):
    def stress_fn(x_): # Das ist damit es mit der jacrev Funktion funktioniert weil die nur 1 Argument erlaubt, dadurch ist r_params fixed für die Iteration
        return first_piola_kirchhoff_stress(x_, r_params) 
    dsdx = torch.func.jacrev(stress_fn)(x)
    residual_x = dsdx[0, 0, 0] + dsdx[0, 1, 1]
    residual_y = dsdx[1, 0, 0] + dsdx[1, 1, 1]
    return residual_x, residual_y

def compute_physics_losses(collocation, top, bottom, left, right, hole, n_hole, r_params):
    # PDE residual
    all_points = torch.cat([collocation, top, bottom, left, right, hole])
    res_x, res_y = torch.vmap(lambda x: pde_residual(x, r_params))(all_points) # Lambda x macht dass r_params fixed ist und x nicht. Ohne das würde es nicht funktionieren weil nicht wie vorher nur 1 Argument in der Funktion ist sondern 2. vmap nur 1 argument
    zeros = torch.zeros_like(res_x, dtype=DTYPE, device=device)
    pde_error = mse(res_x, zeros) + mse(res_y, zeros)

    # left boundary
    pred_left = net(left, r_params)
    bc_left = torch.zeros_like(pred_left[:, 0], dtype=DTYPE, device=device)
    left_error = mse(pred_left[:, 0], bc_left)

    pred_stress_left = torch.vmap(lambda x: first_piola_kirchhoff_stress(x, r_params))(left)
    pred_s_left_xy = pred_stress_left[:, 0, 1]
    s_left_xy = torch.zeros_like(pred_s_left_xy, dtype=DTYPE, device=device)
    left_symm_error = mse(pred_s_left_xy, s_left_xy)

    # bottom boundary
    pred_bottom = net(bottom, r_params)
    bc_bottom = torch.zeros_like(pred_bottom[:, 1], dtype=DTYPE, device=device)
    bottom_error = mse(pred_bottom[:, 1], bc_bottom)

    pred_stress_bottom = torch.vmap(lambda x: first_piola_kirchhoff_stress(x, r_params))(bottom)
    pred_s_bottom_xy = pred_stress_bottom[:, 0, 1]
    s_bottom_xy = torch.zeros_like(pred_s_bottom_xy, dtype=DTYPE, device=device)
    bottom_symm_error = mse(pred_s_bottom_xy, s_bottom_xy)

    # top boundary
    pred_stress_top = torch.vmap(lambda x: first_piola_kirchhoff_stress(x, r_params))(top)
    pred_s_top_yy = pred_stress_top[:, 1, 1]
    pred_s_top_xy = pred_stress_top[:, 0, 1]
    s_top_yy = N2 * torch.ones_like(pred_s_top_yy, dtype=DTYPE, device=device)
    s_top_xy = torch.zeros_like(pred_s_top_xy, dtype=DTYPE, device=device)
    top_error = mse(pred_s_top_yy, s_top_yy) + mse(pred_s_top_xy, s_top_xy)

    # right boundary
    pred_stress_right = torch.vmap(lambda x: first_piola_kirchhoff_stress(x, r_params))(right)
    pred_s_right_xx = pred_stress_right[:, 0, 0]
    pred_s_right_xy = pred_stress_right[:, 0, 1]
    s_right_xx = N1 * torch.ones_like(pred_s_right_xx, dtype=DTYPE, device=device)
    s_right_xy = torch.zeros_like(pred_s_right_xy, dtype=DTYPE, device=device)
    right_error = mse(pred_s_right_xx, s_right_xx) + mse(pred_s_right_xy, s_right_xy)

    # hole boundary
    stress_hole = torch.vmap(lambda x: first_piola_kirchhoff_stress(x, r_params))(hole)
    
    # Calculate traction:
    # Shape: [batch_size, 2, 1]
    traction = torch.einsum("...ij,...j->...i", stress_hole, n_hole)  # Shape: [batch_size, 2]
    
    zeros = torch.zeros_like(traction, dtype=DTYPE, device=device)
    hole_error = mse(traction, zeros)

    return (
        left_error,
        left_symm_error,
        right_error,
        bottom_error,
        bottom_symm_error,
        top_error,
        hole_error,
        pde_error,
    )

# Training loop

In [ ]:
for i in range(Longer_training_factor): 
    progress_bar = tqdm(range(EPOCHS), desc="Training", unit="epoch")

    # Training loop:
    for epoch in progress_bar:
        net.train()
        optimizer.zero_grad()
        
        total_loss_accum = 0.0
        loss_weightless_accum = 0.0
        
        for iteration in range(ITERATIONS):
            # Sample new domain and get r_params
            collocation, top, bottom, left, right, hole, n_hole, Rx, Ry = sample_domain()
            r_params = torch.tensor([[Rx, Ry]], dtype=DTYPE, device=device)  # Shape: [1, 2]
            
            # compute physics losses
            left_err, left_symm_err, right_err, bottom_err, bottom_symm_err, top_err, hole_err, pde_err = compute_physics_losses(
                collocation, top, bottom, left, right, hole, n_hole, r_params
            )
            
            iteration_loss = (
                W_DIR * (left_err + bottom_err) +
                W_NEU * (top_err + right_err + left_symm_err + bottom_symm_err) +
                W_HOLE * hole_err +
                W_PDE * pde_err
            )
            
            iteration_loss_weightless = (
                left_err + bottom_err + top_err + right_err +
                left_symm_err + bottom_symm_err + hole_err + pde_err
            )
            
            # Accumulate losses
            total_loss_accum += iteration_loss
            loss_weightless_accum += iteration_loss_weightless.item()
        
        # Average the losses
        avg_total_loss = total_loss_accum / ITERATIONS
        avg_loss_weightless = loss_weightless_accum / ITERATIONS
        
        # backward + optimize
        avg_total_loss.backward()
        optimizer.step()
        scheduler.step()

        Global_Epoch += 1

        progress_bar.set_postfix(loss_weightless=avg_loss_weightless, global_epoch=Global_Epoch)

    print(f"Global Epoch: {Global_Epoch}")
        

    # Evaluation function
    def evaluate_deeponet(rx, ry, file_path, global_epoch):
        net.eval()
        r_params = torch.tensor([[rx, ry]], dtype=DTYPE, device=device)

        # Extract boundary data
        def extract_ellipse_boundary_data(file_path, tol=0.05, rx=rx, ry=ry):
            data = np.loadtxt(file_path, delimiter=',', skiprows=1)
            ellipse_eq = (data[:, 0]**2 / rx**2) + (data[:, 1]**2 / ry**2)
            mask = np.abs(ellipse_eq - 1) < tol
            points = data[mask, :2]
            stresses = data[mask, 2:5]
            return points, stresses

        points_hole, hole_stress = extract_ellipse_boundary_data(file_path, tol=0.05, rx=rx, ry=ry)
        points_hole_tensor = torch.tensor(points_hole, device=device, dtype=DTYPE)

        with torch.no_grad():
            def compute_stress(x): return sigma(x, r_params)
            stress_hole = torch.vmap(compute_stress)(points_hole_tensor).cpu().numpy()

        sort_idx = np.argsort(points_hole[:, 0])
        points_hole_sorted = points_hole[sort_idx]
        hole_stress_sorted = hole_stress[sort_idx]
        stress_hole_sorted = stress_hole[sort_idx]

        # Plot
        fig = go.Figure()
        line_width = 3
        marker_size = 8

        # σ_xx
        fig.add_trace(go.Scatter(
            x=points_hole[:, 0], y=stress_hole[:, 0, 0],
            marker=dict(color="blue", size=marker_size), mode="markers",
            name="σ_xx (DeepONet)",
        ))
        fig.add_trace(go.Scatter(
            x=points_hole_sorted[:, 0], y=hole_stress_sorted[:, 0],
            line=dict(color="blue", width=line_width, dash='dash'), mode="lines",
            name="σ_xx (FEM)",
        ))

        # σ_yy
        fig.add_trace(go.Scatter(
            x=points_hole[:, 0], y=stress_hole[:, 1, 1],
            marker=dict(color="orange", size=marker_size), mode="markers",
            name="σ_yy (DeepONet)",
        ))
        fig.add_trace(go.Scatter(
            x=points_hole_sorted[:, 0], y=hole_stress_sorted[:, 1],
            line=dict(color="orange", width=line_width, dash='dash'), mode="lines",
            name="σ_yy (FEM)",
        ))

        # σ_xy
        fig.add_trace(go.Scatter(
            x=points_hole[:, 0], y=stress_hole[:, 0, 1],
            marker=dict(color="green", size=marker_size), mode="markers",
            name="σ_xy (DeepONet)",
        ))
        fig.add_trace(go.Scatter(
            x=points_hole_sorted[:, 0], y=hole_stress_sorted[:, 2],
            line=dict(color="green", width=line_width, dash='dash'), mode="lines",
            name="σ_xy (FEM)",
        ))

        fig.update_layout(
            template="plotly_white", width=1600, height=1000,
            title={
                'text': f"Stress Components at Hole Boundary (Rx={rx:.2f}, Ry={ry:.2f}) - Global Epoch: {global_epoch}",
                'y':0.9, 'x':0.5, 'xanchor': 'center', 'yanchor': 'top',
                'font': dict(size=30)
            },
            xaxis=dict(title=dict(text="x-coordinate", font=dict(size=28)), tickfont=dict(size=24)),
            yaxis=dict(title=dict(text="Stress Value", font=dict(size=28)), tickfont=dict(size=24)),
            legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5,
                        itemwidth=80, font=dict(size=20)),
            margin=dict(l=50, r=50, b=50, t=100, pad=4),
            hovermode="x unified"
        )
        fig.show()

        # Error metrics
        mse_xx = np.mean((stress_hole[:, 0, 0] - hole_stress[:, 0])**2)
        mse_yy = np.mean((stress_hole[:, 1, 1] - hole_stress[:, 1])**2)
        mse_xy = np.mean((stress_hole[:, 0, 1] - hole_stress[:, 2])**2)

        print(f"Evaluation for Rx={rx:.1f}, Ry={ry:.1f} (Global Epoch: {global_epoch}):")
        print(f"MSE σ_xx: {mse_xx:.6f}")
        print(f"MSE σ_yy: {mse_yy:.6f}")
        print(f"MSE σ_xy: {mse_xy:.6f}")
        print(f"Total MSE: {mse_xx + mse_yy + mse_xy:.6f}")

        return stress_hole, hole_stress

    # Evaluations
    file_paths = {
        (0.10, 0.10): file_path10,
        (0.25, 0.25): file_path25,
        (0.40, 0.40): file_path40,
    }

    for (rx, ry), file_path in file_paths.items():
        print("="*60)
        deeponet_stress, fem_stress = evaluate_deeponet(rx, ry, file_path, Global_Epoch)

Training: 100%|██████████| 20000/20000 [6:44:21<00:00,  1.21s/epoch, global_epoch=2e+4, loss_weightless=3.68]   

Global Epoch: 20000


Evaluation for Rx=0.1, Ry=0.1 (Global Epoch: 20000):
MSE σ_xx: 23.542254
MSE σ_yy: 2.128250
MSE σ_xy: 3.126603
Total MSE: 28.797106


Evaluation for Rx=0.2, Ry=0.2 (Global Epoch: 20000):
MSE σ_xx: 15.928548
MSE σ_yy: 0.645691
MSE σ_xy: 1.509582
Total MSE: 18.083821


Evaluation for Rx=0.4, Ry=0.4 (Global Epoch: 20000):
MSE σ_xx: 21.599276
MSE σ_yy: 0.803680
MSE σ_xy: 2.358263
Total MSE: 24.761219


...

Training: 100%|██████████| 20000/20000 [12:03:26<00:00,  2.17s/epoch, global_epoch=5e+5, loss_weightless=0.548]     


Global Epoch: 500000


Evaluation for Rx=0.1, Ry=0.1 (Global Epoch: 500000):
MSE σ_xx: 2.937981
MSE σ_yy: 0.215431
MSE σ_xy: 0.344948
Total MSE: 3.498360


Evaluation for Rx=0.2, Ry=0.2 (Global Epoch: 500000):
MSE σ_xx: 0.039449
MSE σ_yy: 0.216659
MSE σ_xy: 0.022452
Total MSE: 0.278560


Evaluation for Rx=0.4, Ry=0.4 (Global Epoch: 500000):
MSE σ_xx: 0.206488
MSE σ_yy: 0.375226
MSE σ_xy: 0.041653
Total MSE: 0.623368


In [ ]:
# same loop twice as there was still improvement after pass 1

for i in range(Longer_training_factor): 
    progress_bar = tqdm(range(EPOCHS), desc="Training", unit="epoch")

    # Training loop:
    for epoch in progress_bar:
        net.train()
        optimizer.zero_grad()
        
        total_loss_accum = 0.0
        loss_weightless_accum = 0.0
        
        for iteration in range(ITERATIONS):
            # Sample new domain and get r_params
            collocation, top, bottom, left, right, hole, n_hole, Rx, Ry = sample_domain()
            r_params = torch.tensor([[Rx, Ry]], dtype=DTYPE, device=device)  # Shape: [1, 2]
            
            # compute physics losses
            left_err, left_symm_err, right_err, bottom_err, bottom_symm_err, top_err, hole_err, pde_err = compute_physics_losses(
                collocation, top, bottom, left, right, hole, n_hole, r_params
            )
            
            iteration_loss = (
                W_DIR * (left_err + bottom_err) +
                W_NEU * (top_err + right_err + left_symm_err + bottom_symm_err) +
                W_HOLE * hole_err +
                W_PDE * pde_err
            )
            
            iteration_loss_weightless = (
                left_err + bottom_err + top_err + right_err +
                left_symm_err + bottom_symm_err + hole_err + pde_err
            )
            
            # Accumulate losses
            total_loss_accum += iteration_loss
            loss_weightless_accum += iteration_loss_weightless.item()
        
        # Average the losses
        avg_total_loss = total_loss_accum / ITERATIONS
        avg_loss_weightless = loss_weightless_accum / ITERATIONS
        
        # backward + optimize
        avg_total_loss.backward()
        optimizer.step()
        scheduler.step()

        Global_Epoch += 1

        progress_bar.set_postfix(loss_weightless=avg_loss_weightless, global_epoch=Global_Epoch)

    print(f"Global Epoch: {Global_Epoch}")
        

    # Evaluation function
    def evaluate_deeponet(rx, ry, file_path, global_epoch):
        net.eval()
        r_params = torch.tensor([[rx, ry]], dtype=DTYPE, device=device)

        # Extract boundary data
        def extract_ellipse_boundary_data(file_path, tol=0.05, rx=rx, ry=ry):
            data = np.loadtxt(file_path, delimiter=',', skiprows=1)
            ellipse_eq = (data[:, 0]**2 / rx**2) + (data[:, 1]**2 / ry**2)
            mask = np.abs(ellipse_eq - 1) < tol
            points = data[mask, :2]
            stresses = data[mask, 2:5]
            return points, stresses

        points_hole, hole_stress = extract_ellipse_boundary_data(file_path, tol=0.05, rx=rx, ry=ry)
        points_hole_tensor = torch.tensor(points_hole, device=device, dtype=DTYPE)

        with torch.no_grad():
            def compute_stress(x): return sigma(x, r_params)
            stress_hole = torch.vmap(compute_stress)(points_hole_tensor).cpu().numpy()

        sort_idx = np.argsort(points_hole[:, 0])
        points_hole_sorted = points_hole[sort_idx]
        hole_stress_sorted = hole_stress[sort_idx]
        stress_hole_sorted = stress_hole[sort_idx]

        # Plot
        fig = go.Figure()
        line_width = 3
        marker_size = 8

        # σ_xx
        fig.add_trace(go.Scatter(
            x=points_hole[:, 0], y=stress_hole[:, 0, 0],
            marker=dict(color="blue", size=marker_size), mode="markers",
            name="σ_xx (DeepONet)",
        ))
        fig.add_trace(go.Scatter(
            x=points_hole_sorted[:, 0], y=hole_stress_sorted[:, 0],
            line=dict(color="blue", width=line_width, dash='dash'), mode="lines",
            name="σ_xx (FEM)",
        ))

        # σ_yy
        fig.add_trace(go.Scatter(
            x=points_hole[:, 0], y=stress_hole[:, 1, 1],
            marker=dict(color="orange", size=marker_size), mode="markers",
            name="σ_yy (DeepONet)",
        ))
        fig.add_trace(go.Scatter(
            x=points_hole_sorted[:, 0], y=hole_stress_sorted[:, 1],
            line=dict(color="orange", width=line_width, dash='dash'), mode="lines",
            name="σ_yy (FEM)",
        ))

        # σ_xy
        fig.add_trace(go.Scatter(
            x=points_hole[:, 0], y=stress_hole[:, 0, 1],
            marker=dict(color="green", size=marker_size), mode="markers",
            name="σ_xy (DeepONet)",
        ))
        fig.add_trace(go.Scatter(
            x=points_hole_sorted[:, 0], y=hole_stress_sorted[:, 2],
            line=dict(color="green", width=line_width, dash='dash'), mode="lines",
            name="σ_xy (FEM)",
        ))

        fig.update_layout(
            template="plotly_white", width=1600, height=1000,
            title={
                'text': f"Stress Components at Hole Boundary (Rx={rx:.2f}, Ry={ry:.2f}) - Global Epoch: {global_epoch}",
                'y':0.9, 'x':0.5, 'xanchor': 'center', 'yanchor': 'top',
                'font': dict(size=30)
            },
            xaxis=dict(title=dict(text="x-coordinate", font=dict(size=28)), tickfont=dict(size=24)),
            yaxis=dict(title=dict(text="Stress Value", font=dict(size=28)), tickfont=dict(size=24)),
            legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5,
                        itemwidth=80, font=dict(size=20)),
            margin=dict(l=50, r=50, b=50, t=100, pad=4),
            hovermode="x unified"
        )
        fig.show()

        # Error metrics
        mse_xx = np.mean((stress_hole[:, 0, 0] - hole_stress[:, 0])**2)
        mse_yy = np.mean((stress_hole[:, 1, 1] - hole_stress[:, 1])**2)
        mse_xy = np.mean((stress_hole[:, 0, 1] - hole_stress[:, 2])**2)

        print(f"Evaluation for Rx={rx:.1f}, Ry={ry:.1f} (Global Epoch: {global_epoch}):")
        print(f"MSE σ_xx: {mse_xx:.6f}")
        print(f"MSE σ_yy: {mse_yy:.6f}")
        print(f"MSE σ_xy: {mse_xy:.6f}")
        print(f"Total MSE: {mse_xx + mse_yy + mse_xy:.6f}")

        return stress_hole, hole_stress

    # Evaluations
    file_paths = {
        (0.10, 0.10): file_path10,
        (0.25, 0.25): file_path25,
        (0.40, 0.40): file_path40,
    }

    for (rx, ry), file_path in file_paths.items():
        print("="*60)
        deeponet_stress, fem_stress = evaluate_deeponet(rx, ry, file_path, Global_Epoch)

Training: 100%|██████████| 20000/20000 [12:14:42<00:00,  2.20s/epoch, global_epoch=520000, loss_weightless=0.509]   


Global Epoch: 520000


Evaluation for Rx=0.1, Ry=0.1 (Global Epoch: 520000):
MSE σ_xx: 2.415367
MSE σ_yy: 0.175634
MSE σ_xy: 0.341999
Total MSE: 2.933000


Evaluation for Rx=0.2, Ry=0.2 (Global Epoch: 520000):
MSE σ_xx: 0.042139
MSE σ_yy: 0.211643
MSE σ_xy: 0.022371
Total MSE: 0.276152


Evaluation for Rx=0.4, Ry=0.4 (Global Epoch: 520000):
MSE σ_xx: 0.155874
MSE σ_yy: 0.336656
MSE σ_xy: 0.034617
Total MSE: 0.527147


...

Global Epoch: 900000


Evaluation for Rx=0.1, Ry=0.1 (Global Epoch: 900000):
MSE σ_xx: 1.018603
MSE σ_yy: 0.193540
MSE σ_xy: 0.194814
Total MSE: 1.406957


Evaluation for Rx=0.2, Ry=0.2 (Global Epoch: 900000):
MSE σ_xx: 0.031294
MSE σ_yy: 0.131249
MSE σ_xy: 0.021375
Total MSE: 0.183918


Evaluation for Rx=0.4, Ry=0.4 (Global Epoch: 900000):
MSE σ_xx: 0.022445
MSE σ_yy: 0.262113
MSE σ_xy: 0.021505
Total MSE: 0.306063


Training:  77%|███████▋  | 15302/20000 [5:03:03<1:33:17,  1.19s/epoch, global_epoch=915302, loss_weightless=0.0561]